In [60]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_groq import ChatGroq
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import operator

In [61]:
load_dotenv()

True

In [62]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [63]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback on the essay")
    score: int = Field(description="Score out of 10")

In [64]:
structured_model = model.with_structured_output(EvaluationSchema)

In [65]:
essay = """
Artificial Intelligence (AI) is about creating machines and programs that can do tasks that usually need human thinking, like learning, recognising patterns, making decisions, and solving problems. We see it in daily life through things like voice assistants, map directions, and online suggestions.
One big benefit of AI is that it works fast and accurately. It can handle a lot of information and help make smarter choices in areas like health, learning, and farming. For example, it helps doctors find diseases early and helps farmers check crops and predict weather.
But there are also worries. Some fear that machines may take jobs, leading to unemployment. Others worry about their personal information being used without permission. There’s also a risk of misuse if there are no rules in place.
Even with these concerns, AI can make life easier by helping with tasks that are too hard, risky, or slow for people. To make sure it’s helpful, clear rules and careful use are important. If used the right way, machines can continue to support people and make life better, while keeping human needs and values first."""

In [66]:
prompt = f"Evaluate the following essay on Artificial Intelligence:\n\n{essay}\n\nProvide detailed feedback and a score out of 10."
structured_model.invoke(prompt).score

7

In [67]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float


In [68]:
def evaluate_language(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the following essay on Artificial Intelligence:\n\n{state['essay']}\n\nProvide detailed feedback and a score out of 10."
    output = structured_model.invoke(prompt)
    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}


In [69]:
def evaluate_analysis(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the depth of analysis in the following essay on Artificial Intelligence:\n\n{state['essay']}\n\nProvide detailed feedback and a score out of 10."
    output = structured_model.invoke(prompt)
    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}


In [70]:
def evaluate_thought(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the clarity of thought in the following essay on Artificial Intelligence:\n\n{state['essay']}\n\nProvide detailed feedback and a score out of 10."
    output = structured_model.invoke(prompt)
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}


In [71]:
def final_evaluation(state: UPSCState) -> UPSCState:
    prompt = f"Based on the following feedback and scores, provide an overall evaluation of the essay on Artificial Intelligence:\n\nLanguage Feedback: {state['language_feedback']}\nAnalysis Feedback: {state['analysis_feedback']}\nClarity Feedback: {state['clarity_feedback']}\nIndividual Scores: {state['individual_scores']}\n\nProvide detailed overall feedback and an average score out of 10."
    overall_feedback = model.invoke(prompt)
    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])
    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [72]:
graph = StateGraph(UPSCState)
graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')
graph.add_edge('final_evaluation', END)

workflow = graph.compile()


In [73]:
initial_state = {
    'essay': essay
}
workflow.invoke(initial_state)

{'essay': '\nArtificial Intelligence (AI) is about creating machines and programs that can do tasks that usually need human thinking, like learning, recognising patterns, making decisions, and solving problems. We see it in daily life through things like voice assistants, map directions, and online suggestions.\nOne big benefit of AI is that it works fast and accurately. It can handle a lot of information and help make smarter choices in areas like health, learning, and farming. For example, it helps doctors find diseases early and helps farmers check crops and predict weather.\nBut there are also worries. Some fear that machines may take jobs, leading to unemployment. Others worry about their personal information being used without permission. There’s also a risk of misuse if there are no rules in place.\nEven with these concerns, AI can make life easier by helping with tasks that are too hard, risky, or slow for people. To make sure it’s helpful, clear rules and careful use are impor